# infer


In [1]:

from melo.api import TTS
from IPython.display import Audio

# Speed is adjustable
speed = 1.0

# CPU is sufficient for real-time inference.
# You can set it manually to 'cpu' or 'cuda' or 'cuda:0' or 'mps'
device = 'cpu'  # Will automatically use GPU if available

# English
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


None


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


None


d:\MeloTTS_Quantization\melo\download_utils.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(ckpt_path, map_location=device)


In [2]:
# American accent
text = """what?"""
output_path = 'en-us.wav'
model.tts_to_file(text, speaker_ids['EN-US'], output_path, speed=speed)


# Load the wav file
audio_file_path = "en-us.wav"

# Display the audio player
Audio(audio_file_path, autoplay=True)

# 23 seconds -> cpu
# 3 seconds -> gpu


 > Text split to sentences.
what?
 > ===========================


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model a

# Convert to ONNX


In [3]:
import re
import torch
import soundfile
import numpy as np
import torch.nn as nn
import torch
from melo import utils
from melo.models import SynthesizerTrn
from melo.split_utils import split_sentence
from melo.download_utils import load_or_download_config, load_or_download_model


def audio_numpy_concat(segment_data_list, sr, speed=1.0):
    audio_segments = []
    for segment_data in segment_data_list:
        audio_segments += segment_data.reshape(-1).tolist()
        audio_segments += [0] * int((sr * 0.05) / speed)
    audio_segments = np.array(audio_segments).astype(np.float32)
    return audio_segments


def split_sentences_into_pieces(text, language, quiet=False):
    texts = split_sentence(text, language_str=language)
    if not quiet:
        print(" > Text split to sentences.")
        print("\n".join(texts))
        print(" > ===========================")
    return texts


device = "cpu"
language = 'EN'
use_hf = True
config_path = None
ckpt_path = None

hps = load_or_download_config(language, use_hf=use_hf, config_path=config_path)
num_languages = hps.num_languages
num_tones = hps.num_tones
symbols = hps.symbols

model = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    num_tones=num_tones,
    num_languages=num_languages,
    **hps.model,
).to(device)

model.eval()
symbol_to_id = {s: i for i, s in enumerate(symbols)}
hps = hps
device = device

# load state_dict
checkpoint_dict = load_or_download_model(
    language, device, use_hf=use_hf, ckpt_path=ckpt_path
)
model.load_state_dict(checkpoint_dict["model"], strict=True)


None


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


None


d:\MeloTTS_Quantization\melo\download_utils.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(ckpt_path, map_location=device)


<All keys matched successfully>

In [4]:
# American accent
text = """
You will be able to kill all your time.
"""
output_path = 'en-us.wav'
speaker_ids = hps.data.spk2id
speaker_id = speaker_ids['EN-US']
output_path = "en-us.wav"
sdp_ratio: float = 0.2
noise_scale: float = 0.6
noise_scale_w: float = 0.8
speed: float = 1.0
quiet: bool = False
texts = split_sentences_into_pieces(text, language, quiet)
audio_list = []
t = texts[0]
t = re.sub(r"([a-z])([A-Z])", r"\1 \2", t)
bert, ja_bert, phones, tones, lang_ids = utils.get_text_for_tts_infer(
    t, language, hps, device, symbol_to_id
)
audio_list = []
with torch.no_grad():
    x_tst = phones.to(device).unsqueeze(0)
    tones = tones.to(device).unsqueeze(0)
    lang_ids = lang_ids.to(device).unsqueeze(0)
    bert = bert.to(device).unsqueeze(0)
    ja_bert = ja_bert.to(device).unsqueeze(0)
    x_tst_lengths = torch.LongTensor([phones.size(0)]).to(device)
    del phones
    speakers = torch.LongTensor([speaker_id]).to(device)


 > Text split to sentences.
You will be able to kill all your time.
 > ===========================


In [5]:
print("x_tst = ", x_tst.dtype)
print("x_tst_lengths = ", x_tst_lengths.dtype)
print("speakers = ", speakers.dtype)
print("tones = ", tones.dtype)
print("lang_ids = ", lang_ids.dtype)
print("bert = ", bert.dtype)
print("ja_bert = ", ja_bert.dtype)


x_tst =  torch.int64
x_tst_lengths =  torch.int64
speakers =  torch.int64
tones =  torch.int64
lang_ids =  torch.int64
bert =  torch.float32
ja_bert =  torch.float32


In [ ]:
torch.onnx.export(
    model,
    (
        x_tst,
        x_tst_lengths,
        speakers,
        tones,
        lang_ids,
        bert,
        ja_bert,
    ),
    "model2.onnx",
    opset_version=11,
    do_constant_folding=True,
    input_names=['x_tst', 'x_tst_lengths', 'speakers',
                 'tones', 'lang_ids', 'bert', 'ja_bert'],
    output_names=['output']
)


In [6]:
out = model(
    x_tst,
    x_tst_lengths,
    speakers,
    tones,
    lang_ids,
    bert,
    ja_bert,
    sdp_ratio=sdp_ratio,
    noise_scale=noise_scale,
    noise_scale_w=noise_scale_w,
    length_scale=1.0 / speed,
)


KeyboardInterrupt: 

In [14]:
out


tensor([-2.4165e-05, -2.5250e-06,  6.5536e-06,  ..., -5.4719e-06,
         6.2125e-06,  1.9549e-05])

In [ ]:

audio = (
    model.infer(
        x_tst,
        x_tst_lengths,
        speakers,
        tones,
        lang_ids,
        bert,
        ja_bert,
        sdp_ratio=sdp_ratio,
        noise_scale=noise_scale,
        noise_scale_w=noise_scale_w,
        length_scale=1.0 / speed,
    )[0][0, 0]
    .data.cpu()
    .float()
    .numpy()
)
del x_tst, tones, lang_ids, bert, ja_bert, x_tst_lengths, speakers

audio_list.append(audio)
torch.cuda.empty_cache()

audio = audio_numpy_concat(
    audio_list, sr=hps.data.sampling_rate, speed=speed
)
soundfile.write(output_path, audio, hps.data.sampling_rate)


In [20]:
from IPython.display import Audio

# Display the audio player
Audio(output_path, autoplay=True)


In [ ]:
import onnx
onnx_model = onnx.load("model2.onnx")
onnx.checker.check_model(onnx_model)


In [11]:
import onnxruntime as ort
import numpy as np

ort_session = ort.InferenceSession("model2.onnx")


Fail: [ONNXRuntimeError] : 1 : FAIL : Load model from model2.onnx failed:Type Error: Type parameter (T) of Optype (Concat) bound to different types (tensor(int64) and tensor(float) in node (/enc_p/encoder/attn_layers.0/Concat_17).

In [1]:
import torch

torch.cuda.is_available()


False

In [2]:
import torch

print(f"Is CUDA supported by this system? {torch.cuda.is_available()}")


Is CUDA supported by this system? False
